# Yield-Curve Construction and Interpolation

## Objective

This notebook constructs two term structures from the same set of observed
continuously compounded zero rates:

1. a term structure obtained by linearly interpolating zero rates;
2. a term structure obtained by linearly interpolating discount factors.

The resulting zero-rate, discount-factor and instantaneous-forward curves are
compared. Later notebooks use both term structures to investigate whether the
interpolation choice affects bond and swap valuation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

In [ ]:
market_data = pd.DataFrame(
    {
        "maturity": [0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0],
        "zero_rate": [0.0420, 0.0435, 0.0445, 0.0438, 0.0415, 0.0400, 0.0390],
    }
)

market_data["discount_factor"] = np.exp(
    -market_data["zero_rate"] * market_data["maturity"]
)

market_data

The inputs are illustrative rather than current market observations. Their
non-flat shape makes differences between interpolation approaches easier to
examine while keeping the experiment controlled.

In [ ]:
curve_maturities = np.linspace(
    market_data["maturity"].min(),
    market_data["maturity"].max(),
    500,
)

## Method 1: Interpolation of zero rates

Let the observed continuously compounded zero rates be \(z_i\) at maturities
\(T_i\). Linear interpolation of the observations produces the zero-rate
function \(z_Z(T)\), where the subscript \(Z\) denotes zero-rate interpolation.

The associated discount curve is

$$
P_Z(0,T)=\exp\left[-Tz_Z(T)\right].
$$

In [ ]:
zero_rate_interpolator = interp1d(
    market_data["maturity"],
    market_data["zero_rate"],
    kind="linear",
)

zero_rates_from_zero_interpolation = zero_rate_interpolator(
    curve_maturities
)

discount_factors_from_zero_interpolation = np.exp(
    -zero_rates_from_zero_interpolation * curve_maturities
)

## Method 2: Interpolation of discount factors

The discount factors observed at the quoted maturities are obtained from

$$
P_i=\exp\left(-T_i z_i\right).
$$

Linear interpolation of the pairs \((T_i,P_i)\) produces the discount curve
\(P_D(0,T)\), where the subscript \(D\) denotes discount-factor interpolation.

The corresponding continuously compounded zero curve is

$$
z_D(T)=-\frac{\log P_D(0,T)}{T}.
$$

In [ ]:
discount_factor_interpolator = interp1d(
    market_data["maturity"],
    market_data["discount_factor"],
    kind="linear",
)

discount_factors_from_discount_interpolation = (
    discount_factor_interpolator(curve_maturities)
)

zero_rates_from_discount_interpolation = (
    -np.log(discount_factors_from_discount_interpolation)
    / curve_maturities
)

## Validation

Both interpolation methods should reproduce the original observations at the
quoted maturities. The resulting discount factors should also remain strictly
positive across the evaluation grid.

In [ ]:
assert np.allclose(
    zero_rate_interpolator(market_data["maturity"]),
    market_data["zero_rate"],
)

assert np.allclose(
    discount_factor_interpolator(market_data["maturity"]),
    market_data["discount_factor"],
)

assert np.all(
    discount_factors_from_zero_interpolation > 0
)

assert np.all(
    discount_factors_from_discount_interpolation > 0
)

print("Validation checks passed.")

## Comparison of interpolated curves

The two approaches are compared through their implied zero-rate and discount
curves. Although they agree at the quoted maturities, they need not agree
between observations.

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    curve_maturities,
    100 * zero_rates_from_zero_interpolation,
    label="Interpolate zero rates",
)

plt.plot(
    curve_maturities,
    100 * zero_rates_from_discount_interpolation,
    linestyle="--",
    label="Interpolate discount factors",
)

plt.scatter(
    market_data["maturity"],
    100 * market_data["zero_rate"],
    label="Observed zero rates",
    zorder=3,
)

plt.xlabel("Maturity (years)")
plt.ylabel("Continuously compounded zero rate (%)")
plt.title("Zero Curves Under Two Interpolation Approaches")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    curve_maturities,
    discount_factors_from_zero_interpolation,
    label="Interpolate zero rates",
)

plt.plot(
    curve_maturities,
    discount_factors_from_discount_interpolation,
    linestyle="--",
    label="Interpolate discount factors",
)

plt.scatter(
    market_data["maturity"],
    market_data["discount_factor"],
    label="Observed discount factors",
    zorder=3,
)

plt.xlabel("Maturity (years)")
plt.ylabel("Discount factor")
plt.title("Discount Curves Under Two Interpolation Approaches")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
comparison = pd.DataFrame(
    {
        "maturity": curve_maturities,
        "zero_rate_interpolation": zero_rates_from_zero_interpolation,
        "discount_factor_interpolation": (
            zero_rates_from_discount_interpolation
        ),
        "discount_factor_from_zero_rates": (
            discount_factors_from_zero_interpolation
        ),
        "interpolated_discount_factor": (
            discount_factors_from_discount_interpolation
        ),
    }
)

comparison["zero_rate_difference_bp"] = (
    comparison["discount_factor_interpolation"]
    - comparison["zero_rate_interpolation"]
) * 10_000

comparison["discount_factor_difference"] = (
    comparison["interpolated_discount_factor"]
    - comparison["discount_factor_from_zero_rates"]
)

comparison.head()

## Quantifying interpolation differences

Differences in the implied zero rates are reported in basis points, while
differences in discount factors are reported in absolute terms.

In [ ]:
maximum_zero_rate_difference_bp = (
    comparison["zero_rate_difference_bp"].abs().max()
)

maximum_discount_factor_difference = (
    comparison["discount_factor_difference"].abs().max()
)

print(
    "Maximum absolute zero-rate difference:",
    f"{maximum_zero_rate_difference_bp:.4f} bp",
)

print(
    "Maximum absolute discount-factor difference:",
    f"{maximum_discount_factor_difference:.8f}",
)

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    comparison["maturity"],
    comparison["zero_rate_difference_bp"],
)

plt.axhline(0, linewidth=1)

plt.xlabel("Maturity (years)")
plt.ylabel("Difference (basis points)")
plt.title(
    "Zero-Rate Difference: Discount-Factor Minus Zero-Rate Interpolation"
)
plt.grid(alpha=0.3)
plt.show()

## Implied instantaneous forward rates

For a discount curve \(P(0,T)\), the instantaneous forward rate is defined by

$$
f(0,T)
=
-\frac{\partial}{\partial T}\log P(0,T).
$$

Applying this definition to each interpolated discount curve yields the
forward curves implied by zero-rate interpolation and discount-factor
interpolation.

The derivative is approximated numerically on the evaluation grid.

In [ ]:
forward_rates_from_zero_interpolation = -np.gradient(
    np.log(discount_factors_from_zero_interpolation),
    curve_maturities,
    edge_order=2,
)

forward_rates_from_discount_interpolation = -np.gradient(
    np.log(discount_factors_from_discount_interpolation),
    curve_maturities,
    edge_order=2,
)

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    curve_maturities,
    100 * forward_rates_from_zero_interpolation,
    label="Interpolate zero rates",
)

plt.plot(
    curve_maturities,
    100 * forward_rates_from_discount_interpolation,
    linestyle="--",
    label="Interpolate discount factors",
)

plt.xlabel("Maturity (years)")
plt.ylabel("Instantaneous forward rate (%)")
plt.title("Implied Forward Curves Under Two Interpolation Approaches")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
forward_rate_difference_bp = (
    forward_rates_from_discount_interpolation
    - forward_rates_from_zero_interpolation
) * 10_000

maximum_forward_rate_difference_bp = np.max(
    np.abs(forward_rate_difference_bp)
)

print(
    "Maximum absolute forward-rate difference:",
    f"{maximum_forward_rate_difference_bp:.4f} bp",
)

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    curve_maturities,
    forward_rate_difference_bp,
)

plt.axhline(0, linewidth=1)

plt.xlabel("Maturity (years)")
plt.ylabel("Difference (basis points)")
plt.title(
    "Forward-Rate Difference: "
    "Discount-Factor Minus Zero-Rate Interpolation"
)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
assert np.all(forward_rates_from_zero_interpolation > 0)
assert np.all(forward_rates_from_discount_interpolation > 0)

print("Forward-rate validation checks passed.")

## Findings

Both interpolation approaches reproduce the observed market inputs exactly at
the quoted maturities. Between those maturities, however, they produce
different term structures because linearity is imposed on different
quantities.

The differences in zero rates and discount factors are relatively small, with
a maximum zero-rate difference of approximately 4.75 basis points. The effect
is considerably more visible in the instantaneous forward curves, where the
maximum absolute difference is approximately 36.85 basis points.

This illustrates that two curves which appear similar when viewed through zero
rates can imply materially different local forward-rate dynamics. The next
stage examines whether these differences propagate into bond prices and
interest-rate risk measures.